In [1]:
!pip -q install "transformers>=4.42.0" "accelerate>=0.31.0" "bitsandbytes>=0.43.0" \
                "langchain>=0.2.0" "langchain-huggingface>=0.0.3" \
                "datasets>=2.19.0" "tqdm" "pandas>=2.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.5 MB/s eta 0:00:00


## 모델 불러오기

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread

model_name = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
streaming = True    # choose the streaming option

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

configuration_exaone.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_exaone.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

## 시스템 프롬프트 & 대화 템플릿 설정

In [5]:
SYSTEM_PROMPT = (
    "You are an AI assistant tasked with solving a question based on a two-person conversation. "
    "Carefully read the dialogue, understand the context, and select the most appropriate answer. "
    "당신은 두 사람의 대화를 바탕으로 문제를 해결하는 AI 어시스턴트입니다. "
    "대화를 주의 깊게 읽고 문맥을 이해한 뒤, 가장 적절한 답을 선택하세요."
)

def build_messages(dialogue: str, question: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"[Dialogue]\n{dialogue}\n\n[Question]\n{question}\n\nInstruction: Output only the single letter of the correct option (A/B/C)."}
    ]

# Llama 3 系열은 <|eot_id|>를 turn 종료로 쓰기도 하므로(eos 외 추가 종료토큰)
# 존재하면 함께 등록 (문서: eos_token_id에 리스트 전달 가능)
terminators = [tokenizer.eos_token_id]
try:
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot_id, int) and eot_id >= 0:
        terminators.append(eot_id)
except Exception:
    pass

## 검증 데이터로 정확도 측정

In [8]:
import re, numpy as np
import pandas as pd
from tqdm import tqdm
import requests

DEV_URL = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/dev.json"
dev_data = requests.get(DEV_URL).json()
df_dev = pd.DataFrame(dev_data)
print("Loaded dev samples:", len(df_dev))

# A/B/C 파싱 유틸
def extract_choice(text: str) -> str:
    if not text:
        return ""
    t = text.strip()
    m = re.search(r"\b([ABC])\b", t)
    if m: return m.group(1)
    m = re.search(r"[정답답안]\s*[:：]\s*([ABC])", t)
    if m: return m.group(1)
    m = re.search(r"[（(]\s*([ABC])\s*[)）]", t)
    if m: return m.group(1)
    for c in "ABC":
        if c in t:
            return c
    return ""

# 카테고리 컬럼명 탐색
possible_cols = ["category", "카테고리", "type", "question_category"]
category_col = next((c for c in possible_cols if c in df_dev.columns), None)

preds, gts, ids, cats, raws = [], [], [], [], []

print("=== dev 데이터셋 평가 시작 ===")
for row in tqdm(df_dev.to_dict(orient="records"), desc="Dev Eval", unit="sample"):
    messages = build_messages(row["dialogue"], row["question"])
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=16,
            eos_token_id=terminators,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    pred = extract_choice(generated)
    gold = row["answer"].strip()
    cat = row.get(category_col, "UNKNOWN") if category_col is not None else "UNKNOWN"

    preds.append(pred)
    gts.append(gold)
    ids.append(row["id"])
    cats.append(cat)
    raws.append(generated)

# 결과 프레임
res = pd.DataFrame({
    "id": ids,
    "category": cats,
    "gold": gts,
    "pred": preds,
    "correct": [int(p == g) for p, g in zip(preds, gts)],
    "raw": raws,
})

# 카테고리별 집계
cat_order = ["후행사건", "동기", "전제", "반응"]
grp = (
    res.groupby("category", dropna=False)
       .agg(n=("correct", "size"), correct=("correct", "sum"))
       .assign(accuracy=lambda d: d["correct"] / d["n"])
)

ordered_index = [c for c in cat_order if c in grp.index] + [c for c in grp.index if c not in cat_order]
grp = grp.loc[ordered_index]

# 전체 집계
overall = pd.DataFrame({
    "n": [int(res.shape[0])],
    "correct": [int(res["correct"].sum())],
    "accuracy": [res["correct"].mean() if res.shape[0] else np.nan],
}, index=["전체"])

# 최종 표
summary_table = pd.concat([grp, overall], axis=0)

# 보기 좋게 퍼센트 표시
display_table = summary_table.copy()
display_table["accuracy"] = (display_table["accuracy"] * 100).round(2).astype(str) + "%"

print("\n=== dev 카테고리별 및 전체 정확도 ===")
display(display_table)

correct = int(res["correct"].sum())
total = int(res.shape[0])
print(f"\n[dev] Overall Accuracy: {correct}/{total} = {correct/total:.4f}")

Loaded dev samples: 151
=== dev 데이터셋 평가 시작 ===


Dev Eval:   0%|          | 0/151 [00:00<?, ?sample/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Dev Eval: 100%|██████████| 151/151 [00:43<00:00,  3.46sample/s]


=== dev 카테고리별 및 전체 정확도 ===


,n,correct,accuracy
후행사건,51,40,78.43%
동기,25,22,88.0%
전제,25,23,92.0%
반응,25,20,80.0%
원인,25,20,80.0%
전체,151,125,82.78%



[dev] Overall Accuracy: 125/151 = 0.8278


## 테스트 데이터로 제출물 생성

In [9]:
import json, pandas as pd, requests

RAW_URL = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/test.json"
data = requests.get(RAW_URL).json()
df = pd.DataFrame(data)

from tqdm import tqdm

mapping = {
    "A": "inference_1",
    "B": "inference_2",
    "C": "inference_3"
}

results = []

for row in tqdm(df.to_dict(orient="records"), desc="Processing", unit="sample"):
    messages = build_messages(row["dialogue"], row["question"])
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=16,
            eos_token_id=terminators,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    choice = None
    for ch in ["A", "B", "C"]:
        if ch in generated:
            choice = ch
            break

    if choice is None:
        tqdm.write(f"[경고] {row['id']}에서 답을 찾지 못함 → '{generated}'")
        continue

    results.append({
        "id": row["id"],
        "output": mapping[choice]
    })

output_file = "inference_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"총 {len(results)}개 결과 저장 완료 → {output_file}")


Processing: 100%|██████████| 605/605 [02:46<00:00,  3.64sample/s]

총 605개 결과 저장 완료 → inference_results.json
